In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="trendcast",
    user="phamphuongkhanh",
    password="",
    host="localhost",
    port=5432
)
cur = conn.cursor()
print("Connected!")

Connected!


In [ ]:
cur.execute("""
CREATE TABLE IF NOT EXISTS companies (
    company_id      SERIAL PRIMARY KEY,
    company_name    VARCHAR(100) NOT NULL,
    ticker_symbol   VARCHAR(10) UNIQUE NOT NULL,
    industry        VARCHAR(100),
    sector          VARCHAR(100),
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS filings (
    filing_id         SERIAL PRIMARY KEY,
    company_id        INT REFERENCES companies(company_id),
    filing_type       VARCHAR(10) NOT NULL,
    filing_date       DATE NOT NULL,
    period_of_report  DATE,
    accession_number  VARCHAR(25) UNIQUE
);

CREATE TABLE IF NOT EXISTS financials (
    financial_id   SERIAL PRIMARY KEY,
    filing_id      INT REFERENCES filings(filing_id),
    company_id     INT REFERENCES companies(company_id),
    revenue        NUMERIC(20, 2),
    net_income     NUMERIC(20, 2),
    eps            NUMERIC(10, 4),
    rd_spend       NUMERIC(20, 2),
    fiscal_year    INT NOT NULL,
    fiscal_quarter INT CHECK (fiscal_quarter BETWEEN 1 AND 4)
);

CREATE TABLE IF NOT EXISTS trend_scores (
    score_id     SERIAL PRIMARY KEY,
    company_id   INT REFERENCES companies(company_id),
    score_date   DATE NOT NULL,
    trend_score  NUMERIC(5, 2),
    keyword      VARCHAR(100),
    source       VARCHAR(50)
);
""")

conn.commit()
print("Tables created!")

Tables created!


In [ ]:
cur.execute("""
INSERT INTO companies (company_name, ticker_symbol, industry, sector) VALUES
    ('Apple Inc.',        'AAPL', 'Consumer Electronics', 'Technology'),
    ('Sony Group',        'SONY', 'Consumer Electronics', 'Technology'),
    ('Dell Technologies', 'DELL', 'Computer Hardware',    'Technology'),
    ('HP Inc.',           'HPQ',  'Computer Hardware',    'Technology'),
    ('Garmin Ltd.',       'GRMN', 'Wearables & GPS',      'Technology'),
    ('NVIDIA Corp.',      'NVDA', 'Semiconductors',       'Technology')
ON CONFLICT (ticker_symbol) DO NOTHING;
""")

conn.commit()
print("Companies inserted!")

Companies inserted!


In [ ]:
cur.execute("""
CREATE OR REPLACE VIEW dashboard_view AS
SELECT
    ts.keyword,

    CASE
        WHEN ts.keyword IN ('Apple', 'NVIDIA', 'Samsung', 'Sony', 'Microsoft', 'Intel', 'AMD', 'Logitech')
            THEN 'company'
        WHEN ts.keyword IN ('smartphone', 'laptop', 'tablet', 'smartwatch', 'earbuds', 'monitor', 'smart TV')
            THEN 'categorical'
        ELSE 'feature'
    END AS keyword_group,

    ts.trend_score,
    ts.score_date AS updated_at,
    NULL::NUMERIC AS sentiment_score,
    NULL::INT     AS news_count,
    NULL::NUMERIC AS google_trends_score,
    f.revenue,
    f.eps,
    f.net_income

FROM trend_scores ts

LEFT JOIN companies c
    ON LOWER(ts.keyword) = LOWER(SPLIT_PART(c.company_name, ' ', 1))

LEFT JOIN financials f
    ON f.company_id = c.company_id
    AND f.fiscal_year = EXTRACT(YEAR FROM ts.score_date)::INT;
""")

conn.commit()
print("Dashboard view created!")

Dashboard view created!


In [ ]:
import requests

HEADERS = {"User-Agent": "TrendCast phamphuongkhanh@email.com"}

companies = {
    "AAPL": {"name": "Apple Inc.",        "cik": "0000320193"},
    "SONY": {"name": "Sony Group",        "cik": "0000318154"},
    "DELL": {"name": "Dell Technologies", "cik": "0001571123"},
    "HPQ":  {"name": "HP Inc.",           "cik": "0000047217"},
    "GRMN": {"name": "Garmin Ltd.",       "cik": "0001121788"},
    "NVDA": {"name": "NVIDIA Corp.",      "cik": "0001045810"},
}

print("Ready to scrape!")

Ready to scrape!


In [ ]:
def get_financials(cik, ticker, name):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code != 200:
        print(f"  [FAILED] {name}")
        return []

    facts = resp.json().get("facts", {}).get("us-gaap", {})
    results = []

    # Revenue
    revenue_entries = []
    for key in ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"]:
        if key in facts:
            entries = facts[key]["units"]["USD"]
            revenue_entries = [e for e in entries if e.get("form") == "10-K"]
            if revenue_entries:
                break

    # EPS
    eps_entries = []
    for key in ["EarningsPerShareBasic", "EarningsPerShareDiluted"]:
        if key in facts:
            entries = facts[key]["units"]["USD/shares"]
            eps_entries = [e for e in entries if e.get("form") == "10-K"]
            if eps_entries:
                break

    # Net Income
    net_income_entries = []
    if "NetIncomeLoss" in facts:
        entries = facts["NetIncomeLoss"]["units"]["USD"]
        net_income_entries = [e for e in entries if e.get("form") == "10-K"]

    # Match by date
    for entry in revenue_entries[-4:]:
        end_date = entry.get("end")
        fiscal_year = int(end_date[:4])
        eps_val = next((e["val"] for e in eps_entries if e.get("end") == end_date), None)
        ni_val  = next((e["val"] for e in net_income_entries if e.get("end") == end_date), None)

        results.append({
            "ticker":      ticker,
            "name":        name,
            "end_date":    end_date,
            "fiscal_year": fiscal_year,
            "revenue":     entry.get("val"),
            "eps":         eps_val,
            "net_income":  ni_val,
            "accession":   entry.get("accn"),
        })
        print(f"  {end_date} | Revenue: ${entry.get('val'):,.0f} | EPS: {eps_val} | NI: {ni_val}")

    return results

all_data = []
for ticker, info in companies.items():
    print(f"\n=== {info['name']} ({ticker}) ===")
    rows = get_financials(info["cik"], ticker, info["name"])
    all_data.extend(rows)

print(f"\nTotal rows collected: {len(all_data)}")


=== Apple Inc. (AAPL) ===
  2018-03-31 | Revenue: $61,137,000,000 | EPS: 2.75 | NI: 13822000000
  2018-06-30 | Revenue: $53,265,000,000 | EPS: 2.36 | NI: 11519000000
  2018-09-29 | Revenue: $265,595,000,000 | EPS: 12.01 | NI: 59531000000
  2018-09-29 | Revenue: $62,900,000,000 | EPS: 12.01 | NI: 59531000000

=== Sony Group (SONY) ===
  2018-12-31 | Revenue: $23,747,000,000 | EPS: 12.7 | NI: 8394000000
  2019-12-31 | Revenue: $23,362,000,000 | EPS: 12.96 | NI: 7842000000
  2019-12-31 | Revenue: $23,362,000,000 | EPS: 12.96 | NI: 7842000000
  2020-12-31 | Revenue: $25,424,000,000 | EPS: 12.4 | NI: 7264000000

=== Dell Technologies (DELL) ===
  2024-02-02 | Revenue: $7,444,000,000 | EPS: 8.98 | NI: 477000000
  2025-01-31 | Revenue: $7,479,000,000 | EPS: 7.23 | NI: 362000000
  2025-01-31 | Revenue: $7,479,000,000 | EPS: 7.23 | NI: 362000000
  2026-01-30 | Revenue: $7,262,000,000 | EPS: 7.73 | NI: None

=== HP Inc. (HPQ) ===
  2023-10-31 | Revenue: $53,718,000,000 | EPS: 3.29 | NI: 3263000

In [ ]:
for row in all_data:
    # Get company_id
    cur.execute("SELECT company_id FROM companies WHERE ticker_symbol = %s", (row["ticker"],))
    result = cur.fetchone()
    if not result:
        print(f"Company not found: {row['ticker']}")
        continue
    company_id = result[0]

    # Insert into filings
    cur.execute("""
        INSERT INTO filings (company_id, filing_type, filing_date, period_of_report, accession_number)
        VALUES (%s, '10-K', %s, %s, %s)
        ON CONFLICT (accession_number) DO NOTHING
        RETURNING filing_id
    """, (company_id, row["end_date"], row["end_date"], row["accession"]))

    filing_row = cur.fetchone()
    if not filing_row:
        cur.execute("SELECT filing_id FROM filings WHERE accession_number = %s", (row["accession"],))
        filing_row = cur.fetchone()
    filing_id = filing_row[0]

    # Insert into financials
    cur.execute("""
        INSERT INTO financials (filing_id, company_id, revenue, net_income, eps, fiscal_year)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (filing_id, company_id, row["revenue"], row["net_income"], row["eps"], row["fiscal_year"]))

    print(f"  Loaded: {row['name']} | {row['end_date']} | ${row['revenue']:,.0f}")

conn.commit()
print("\nAll data loaded into PostgreSQL!")

  Loaded: Apple Inc. | 2018-03-31 | $61,137,000,000
  Loaded: Apple Inc. | 2018-06-30 | $53,265,000,000
  Loaded: Apple Inc. | 2018-09-29 | $265,595,000,000
  Loaded: Apple Inc. | 2018-09-29 | $62,900,000,000
  Loaded: Sony Group | 2018-12-31 | $23,747,000,000
  Loaded: Sony Group | 2019-12-31 | $23,362,000,000
  Loaded: Sony Group | 2019-12-31 | $23,362,000,000
  Loaded: Sony Group | 2020-12-31 | $25,424,000,000
  Loaded: Dell Technologies | 2024-02-02 | $7,444,000,000
  Loaded: Dell Technologies | 2025-01-31 | $7,479,000,000
  Loaded: Dell Technologies | 2025-01-31 | $7,479,000,000
  Loaded: Dell Technologies | 2026-01-30 | $7,262,000,000
  Loaded: HP Inc. | 2023-10-31 | $53,718,000,000
  Loaded: HP Inc. | 2024-10-31 | $53,559,000,000
  Loaded: HP Inc. | 2024-10-31 | $53,559,000,000
  Loaded: HP Inc. | 2025-10-31 | $55,295,000,000
  Loaded: Garmin Ltd. | 2023-12-30 | $5,228,252,000
  Loaded: Garmin Ltd. | 2024-12-28 | $6,296,903,000
  Loaded: Garmin Ltd. | 2024-12-28 | $6,296,903,000